# LeRobot Air Hockey on Google Colab

Run AI-controlled air hockey with GPU acceleration on Colab.

In [ ]:
# Install LeRobot and dependencies
!pip install lerobot
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip install huggingface_hub

In [ ]:
# Authenticate with Hugging Face (if needed)
from huggingface_hub import login
login()  # Will prompt for token

In [ ]:
# Download and set up the migrated model
from lerobot.policies.act.configuration_act import ACTConfig
from lerobot.policies.factory import make_policy

# Load the migrated model from Hugging Face
config = ACTConfig.from_pretrained("AIBunCho/air-hockey-5000")  # Use smaller model for testing
policy = make_policy(config)
policy.eval()
print("Model loaded successfully!")

In [ ]:
# Test inference speed on GPU
import torch
import time

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
policy.to(device)
print(f"Using device: {device}")

# Create test input
batch = {
    'observation.images.front': torch.randn(1, 3, 1080, 1920).to(device),
    'observation.state': torch.randn(1, 6).to(device)
}

# Warm up
with torch.no_grad():
    _ = policy(batch)

# Time inference
times = []
for i in range(10):
    start = time.time()
    with torch.no_grad():
        action = policy(batch)
    end = time.time()
    times.append(end - start)
    print(f"Inference {i+1}: {end - start:.3f}s")

avg_time = sum(times) / len(times)
fps = 1.0 / avg_time
print(f"\nAverage inference time: {avg_time:.3f}s")
print(f"Inference FPS: {fps:.1f}")
print(f"Expected control FPS: {min(fps, 30):.1f}")  # Limited by camera

## Expected Performance on Colab GPU:

- **A100**: 50-100 FPS inference
- **V100/T4**: 20-50 FPS inference  
- **Your MacBook CPU**: 2-5 FPS inference

This should give much more responsive robot control!

## For Real Robot Control:

To control your physical SO-101 robot from Colab, you would need:

1. **Network streaming**: Set up a way to stream video from your iPhone to Colab
2. **Remote robot control**: Network connection to your robot
3. **Real-time communication**: Low-latency connection between Colab and your robot

This is more complex and would require additional networking setup.